# FinDisputeEval - CFPB Seed Selection v4 (review-aware overlay)

v4 does not re-download or re-score the CFPB corpus. It consumes the completed the curated v3 release plus review round 1 and produces a clean `dataset/curated/seed_pools/cfpb_dispute/seed_v04/` release.

## v4 changes

1. Correct the Zelle sanity anchor universe: `~0.13` is the pre-dedup diagnostic; `0.0586` is the deduped, weighted section-8.13 reference baseline.
2. Apply manual relabel decisions where the row was not blocked by the low-q50 quality audit.
3. Exclude `questionable` and `bad_seed` low-q50 reviews from the formal evaluation seed pool.
4. Preserve excluded and reroute-candidate rows in audit logs instead of silently deleting them.

## Review policy

| review signal | v4 action |
|---|---|
| `quality_ok` | keep in formal seed |
| `questionable` | exclude from formal evaluation seed; keep in review log |
| `bad_seed` | exclude from current formal seed; keep suggested reroutes as candidates only |
| `relabel_decision=change_claim` without quality exclusion | apply suggested label fields |
| `relabel_decision=keep_current` | keep current label |


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
from copy import deepcopy
import csv, json, shutil, hashlib, sys


def progress(msg: str):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)


## Step 1 - Environment and paths


In [ ]:
IN_COLAB = "google.colab" in sys.modules


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "dataset").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the FinDisputeEval project root.")


if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/FinDisputeEval")
else:
    PROJECT_ROOT = find_project_root(Path.cwd())

V3_DIR = PROJECT_ROOT / "dataset" / "curated" / "seed_pools" / "cfpb_dispute" / "seed_v03"
REVIEW_DIR = PROJECT_ROOT / "dataset" / "curated" / "annotations" / "cfpb_seed_review" / "round_001"
OUT_DIR = PROJECT_ROOT / "dataset" / "curated" / "seed_pools" / "cfpb_dispute" / "seed_v04"

OUT_DIR.mkdir(parents=True, exist_ok=True)
progress(f"Step 1/7 complete: V3_DIR={V3_DIR}; REVIEW_DIR={REVIEW_DIR}; OUT_DIR={OUT_DIR}")


## Step 2 - Helpers and input checks


In [ ]:
def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def load_jsonl(p: Path):
    with p.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def write_jsonl(p: Path, rows):
    with p.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def write_csv(p: Path, rows, fieldnames):
    with p.open("w", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)


def parse_bool(v, default=False):
    if isinstance(v, bool):
        return v
    s = str(v).strip().lower()
    if s in {"true", "1", "yes", "y"}:
        return True
    if s in {"false", "0", "no", "n"}:
        return False
    return default


def cell_dist(rows):
    return dict(sorted(Counter(f"{r['card_type']}|{r['claim_type']}" for r in rows).items()))


paths = {
    "v3_seed": V3_DIR / "cfpb_seed_pool.jsonl",
    "v3_ref": V3_DIR / "cfpb_reference_distributions.json",
    "v3_manifest": V3_DIR / "seed_selection_manifest.json",
    "schema": V3_DIR / "scenario_seed_candidate.schema.json",
    "oos": V3_DIR / "oos_negative_pool.jsonl",
    "lid": V3_DIR / "lid_handoff_probe.jsonl",
    "review_quality": REVIEW_DIR / "manual_audit_sample_quality_reviewed.csv",
    "review_relabel": REVIEW_DIR / "manual_audit_sample_relabel_reviewed.csv",
}
missing = [str(p) for p in paths.values() if not p.exists()]
if missing:
    raise FileNotFoundError("Missing v4 inputs:\n" + "\n".join(missing))
progress("Step 2/7 complete: all v3/review inputs found")


## Step 3 - Load manual review decisions


In [ ]:
progress("Step 3/7: loading manual review decisions")
review_rows = []
with paths["review_quality"].open(encoding="utf-8-sig", newline="") as f:
    review_rows.extend(csv.DictReader(f))

review = {}
conflicts = []
for row in review_rows:
    sid = row.get("seed_id", "").strip()
    if not sid:
        continue
    rec = review.setdefault(sid, {
        "seed_id": sid,
        "complaint_id": row.get("complaint_id", ""),
        "audit_groups": [],
        "quality_audit_label": "",
        "quality_action": "",
        "quality_reason": "",
        "relabel_decision": "",
        "suggested_card_type": "",
        "suggested_claim_type": "",
        "suggested_route_hint": "",
        "suggested_relabel_required": "",
        "relabel_reason": "",
    })
    ag = row.get("audit_group", "").strip()
    if ag and ag not in rec["audit_groups"]:
        rec["audit_groups"].append(ag)
    for k in ["quality_audit_label", "quality_action", "quality_reason"]:
        val = row.get(k, "").strip()
        if val:
            if rec[k] and rec[k] != val:
                conflicts.append((sid, k, rec[k], val))
            rec[k] = val
    decision = row.get("relabel_decision", "").strip()
    if decision:
        if rec["relabel_decision"] and rec["relabel_decision"] != decision:
            conflicts.append((sid, "relabel_decision", rec["relabel_decision"], decision))
        rec["relabel_decision"] = decision
        for k in ["suggested_card_type", "suggested_claim_type", "suggested_route_hint", "suggested_relabel_required", "relabel_reason"]:
            val = row.get(k, "").strip()
            if val:
                if rec[k] and rec[k] != val:
                    conflicts.append((sid, k, rec[k], val))
                rec[k] = val

for rec in review.values():
    rec["audit_groups"] = "|".join(rec["audit_groups"])
if conflicts:
    raise ValueError(f"Review conflicts detected: {conflicts[:5]}")

print("reviewed_unique_seed_ids", len(review))
print("quality labels", Counter(v.get("quality_audit_label") or "blank" for v in review.values()))
print("relabel decisions", Counter(v.get("relabel_decision") or "blank" for v in review.values()))
progress("Step 3/7 complete")


## Step 4 - Apply review overlay to v3 seed pool


In [ ]:
progress("Step 4/7: applying review-aware seed overlay")
seeds_v3 = load_jsonl(paths["v3_seed"])
formal = []
excluded = []
reroute_candidates = []
overlay_rows = []
action_counts = Counter()

for rec in seeds_v3:
    sid = rec["seed_id"]
    rev = review.get(sid)
    final = deepcopy(rec)
    final_action = "keep_unreviewed"
    final_reason = ""

    if rev:
        qlabel = rev.get("quality_audit_label", "")
        rdecision = rev.get("relabel_decision", "")
        if qlabel in {"bad_seed", "questionable"}:
            final_action = f"exclude_{qlabel}"
            final_reason = rev.get("quality_reason") or rev.get("relabel_reason")
            excluded_item = deepcopy(rec)
            excluded_item["manual_review"] = rev
            excluded_item["final_action"] = final_action
            excluded.append(excluded_item)
            if rdecision == "change_claim" and rev.get("suggested_claim_type"):
                cand = deepcopy(rec)
                cand["suggested_card_type"] = rev.get("suggested_card_type") or rec["card_type"]
                cand["suggested_claim_type"] = rev.get("suggested_claim_type")
                cand["suggested_route_hint"] = rev.get("suggested_route_hint") or rec["route_hint"]
                cand["suggested_relabel_required"] = parse_bool(rev.get("suggested_relabel_required"), rec.get("relabel_required", False))
                cand["manual_review"] = rev
                cand["candidate_note"] = "Excluded from formal v4 seed because quality audit was not quality_ok; keep only as reroute candidate."
                reroute_candidates.append(cand)
        elif rdecision == "change_claim":
            final["card_type"] = rev.get("suggested_card_type") or final["card_type"]
            final["claim_type"] = rev.get("suggested_claim_type") or final["claim_type"]
            final["route_hint"] = rev.get("suggested_route_hint") or final["route_hint"]
            final["relabel_required"] = parse_bool(rev.get("suggested_relabel_required"), final["relabel_required"])
            final_action = "relabel_changed"
            final_reason = rev.get("relabel_reason", "")
            formal.append(final)
        elif rdecision == "keep_current":
            final_action = "review_keep_current"
            final_reason = rev.get("relabel_reason", "")
            formal.append(final)
        elif qlabel == "quality_ok":
            final_action = "quality_ok_keep"
            final_reason = rev.get("quality_reason", "")
            formal.append(final)
        else:
            final_action = "review_no_action"
            formal.append(final)
    else:
        formal.append(final)

    if rev:
        overlay_rows.append({
            "seed_id": sid,
            "complaint_id": rec.get("complaint_id", ""),
            "audit_groups": rev.get("audit_groups", ""),
            "original_card_type": rec.get("card_type", ""),
            "original_claim_type": rec.get("claim_type", ""),
            "original_route_hint": rec.get("route_hint", ""),
            "original_relabel_required": rec.get("relabel_required", ""),
            "quality_audit_label": rev.get("quality_audit_label", ""),
            "quality_action": rev.get("quality_action", ""),
            "relabel_decision": rev.get("relabel_decision", ""),
            "suggested_card_type": rev.get("suggested_card_type", ""),
            "suggested_claim_type": rev.get("suggested_claim_type", ""),
            "suggested_route_hint": rev.get("suggested_route_hint", ""),
            "suggested_relabel_required": rev.get("suggested_relabel_required", ""),
            "final_action": final_action,
            "final_card_type": final.get("card_type", ""),
            "final_claim_type": final.get("claim_type", ""),
            "final_route_hint": final.get("route_hint", ""),
            "final_relabel_required": final.get("relabel_required", ""),
            "review_reason": final_reason,
        })
    action_counts[final_action] += 1

if len({r["seed_id"] for r in formal}) != len(formal):
    raise ValueError("Duplicate seed_id created in formal v4 seed pool")

seed_path = OUT_DIR / "cfpb_seed_pool.jsonl"
excluded_path = OUT_DIR / "cfpb_seed_pool_review_excluded.jsonl"
reroute_path = OUT_DIR / "cfpb_seed_pool_reroute_candidates.jsonl"
overlay_path = OUT_DIR / "manual_review_overlay.csv"
summary_path = OUT_DIR / "v4_review_summary.json"

write_jsonl(seed_path, formal)
write_jsonl(excluded_path, excluded)
write_jsonl(reroute_path, reroute_candidates)
write_csv(overlay_path, overlay_rows, [
    "seed_id", "complaint_id", "audit_groups", "original_card_type", "original_claim_type", "original_route_hint", "original_relabel_required",
    "quality_audit_label", "quality_action", "relabel_decision", "suggested_card_type", "suggested_claim_type", "suggested_route_hint", "suggested_relabel_required",
    "final_action", "final_card_type", "final_claim_type", "final_route_hint", "final_relabel_required", "review_reason",
])

summary = {
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "notebook_version": 4,
    "source_seed_rows": len(seeds_v3),
    "formal_seed_rows": len(formal),
    "reviewed_unique_seed_ids": len(review),
    "excluded_rows": len(excluded),
    "reroute_candidate_rows": len(reroute_candidates),
    "action_counts": dict(action_counts),
    "quality_label_counts_reviewed": dict(Counter(v.get("quality_audit_label") or "blank" for v in review.values())),
    "relabel_decision_counts_reviewed": dict(Counter(v.get("relabel_decision") or "blank" for v in review.values())),
    "cell_distribution_before": cell_dist(seeds_v3),
    "cell_distribution_after": cell_dist(formal),
    "review_policy": {
        "quality_ok": "keep in formal seed",
        "questionable": "exclude from formal evaluation seed; retain in review_excluded log",
        "bad_seed": "exclude from current formal seed; if relabel suggestion exists, retain only in reroute_candidates",
        "relabel_change_without_quality_exclusion": "apply suggested card_type/claim_type/route_hint/relabel_required",
    },
}
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: summary[k] for k in ["source_seed_rows", "formal_seed_rows", "reviewed_unique_seed_ids", "excluded_rows", "reroute_candidate_rows", "action_counts"]}, ensure_ascii=False, indent=2))
progress("Step 4/7 complete")


## Step 5 - Correct reference Zelle sanity anchor


In [ ]:
progress("Step 5/7: writing corrected reference distributions")
ref = json.loads(paths["v3_ref"].read_text(encoding="utf-8"))
ref["notebook_version"] = 4
ref["generated_utc"] = datetime.now(timezone.utc).isoformat()
pop = ref.setdefault("population_proxy_inscope_2025plus", {})
pop["zelle_sanity_anchor"] = {
    "interpretation": "The earlier ~0.13 anchor is a pre-dedup diagnostic, not the section-8.13 reference universe.",
    "layered_local_replication": [
        {"gate": "raw claim rows before quality gates", "n": 121001, "zelle_rate": 0.127},
        {"gate": "token length 60-800", "n": 106652, "zelle_rate": 0.133},
        {"gate": "mask density <= 0.25", "n": 105822, "zelle_rate": 0.134},
        {"gate": "template-family dedup", "n": 73456, "zelle_rate": 0.064},
    ],
    "pipeline_weighted_reference": {
        "estimated_population_rows": pop.get("estimated_population_rows"),
        "zelle_rate": pop.get("phenomenon_rates", {}).get("zelle_mention"),
    },
    "decision": "Use the deduped, weighted pipeline value as the final section-8.13 baseline; retain ~0.13 only as a pre-dedup diagnostic.",
}
pop["note"] = (
    "Population proxy for 2025-01+ x four in-scope products. Selected by product x date criteria "
    "(source-independent) and re-weighted by feature-pool inclusion probability to undo zelle/prepaid protection oversampling. "
    "Use this block for the section-8.13 KL/JS gate. Zelle sanity anchor clarification: the earlier ~0.13 anchor refers "
    "to the pre-dedup claim universe. Local gate decomposition gives 0.127 raw, 0.133 after token gate, 0.134 after mask-density gate, "
    "and 0.064 after template-family dedup. This pipeline estimates zelle_mention=0.0586 over the deduped weighted reference universe, "
    "so 0.0586 is the v4 baseline. The drop indicates Zelle-mentioning complaints are disproportionately template-like or near-duplicate; "
    "synthetic generation should avoid amplifying residual template phrasing."
)
pop["dedup_finding"] = (
    "Zelle-mentioning claim rows survive template-family dedup at about 31% versus about 65% for non-Zelle rows. "
    "This is a corpus limitation and generation-risk note, not a math bug."
)
ref_path = OUT_DIR / "cfpb_reference_distributions.json"
ref_path.write_text(json.dumps(ref, ensure_ascii=False, indent=2), encoding="utf-8")
print("population zelle baseline", pop.get("phenomenon_rates", {}).get("zelle_mention"))
progress("Step 5/7 complete")


## Step 6 - Manifest and stable artifacts


In [ ]:
progress("Step 6/7: copying stable artifacts and writing manifest")
copy_map = {
    paths["schema"]: OUT_DIR / "scenario_seed_candidate.schema.json",
    paths["oos"]: OUT_DIR / "oos_negative_pool.jsonl",
    paths["lid"]: OUT_DIR / "lid_handoff_probe.jsonl",
    paths["review_quality"]: OUT_DIR / "manual_audit_sample_quality_reviewed.csv",
    paths["review_relabel"]: OUT_DIR / "manual_audit_sample_relabel_reviewed.csv",
}
for src, dst in copy_map.items():
    shutil.copy2(src, dst)

v3_manifest = json.loads(paths["v3_manifest"].read_text(encoding="utf-8"))
manifest = {
    "notebook": "FinDisputeEval_CFPB_SeedPool_ReviewOverlay_colab_v04",
    "notebook_version": 4,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "parent_notebook_version": v3_manifest.get("notebook_version"),
    "parent_manifest_sha256": sha256_file(paths["v3_manifest"]),
    "seed": v3_manifest.get("seed", 20260703),
    "version_changes": [
        "v4: review-aware overlay built from v3 outputs, without re-downloading CFPB data",
        "v4: corrected Zelle sanity anchor universe; 0.13 is pre-dedup diagnostic, 0.0586 is section-8.13 baseline",
        "v4: applied manual relabel decisions where not blocked by quality audit",
        "v4: excluded questionable/bad_seed low-q50 reviews from formal evaluation seed pool",
        "v4: retained excluded/reroute candidates in separate audit logs",
    ],
    "inputs": {name: {"path": str(path), "sha256": sha256_file(path)} for name, path in paths.items()},
    "outputs": {},
    "stages": {
        "source_seed_rows": len(seeds_v3),
        "reviewed_unique_seed_ids": len(review),
        "review_quality_excluded_rows": len(excluded),
        "review_reroute_candidate_rows": len(reroute_candidates),
        "seed_rows": len(formal),
    },
    "review_policy": summary["review_policy"],
    "review_action_counts": dict(action_counts),
    "cell_distribution_after": summary["cell_distribution_after"],
}
for p in [seed_path, excluded_path, reroute_path, overlay_path, summary_path, ref_path, *copy_map.values()]:
    meta = {"sha256": sha256_file(p)}
    if p.suffix == ".jsonl":
        meta["rows"] = sum(1 for line in p.open(encoding="utf-8") if line.strip())
    elif p.suffix == ".csv":
        with p.open(encoding="utf-8-sig", newline="") as f:
            meta["rows"] = sum(1 for _ in csv.DictReader(f))
    manifest["outputs"][p.name] = meta
manifest_path = OUT_DIR / "seed_selection_manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(manifest["stages"], ensure_ascii=False, indent=2))
progress("Step 6/7 complete")


## Step 7 - Output summary


In [ ]:
progress("Step 7/7: v4 summary")
print("v4 outputs:")
for p in sorted(OUT_DIR.iterdir()):
    print(f"  {p.name:45s} {p.stat().st_size:>10,} bytes")
print("\nformal seed rows:", len(formal))
print("excluded rows:", len(excluded))
print("reroute candidate rows:", len(reroute_candidates))
print("\naction counts:")
for k, v in action_counts.items():
    print(f"  {k}: {v}")
progress("Step 7/7 complete")
